# Rule Testing

This notebook tests individual rules with synthetic symbol sets, shows inference chains, and validates rule logic.

In [1]:
import sys
from pathlib import Path

# Add src to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

from src.knowledge_base.rules import EconomicTheoryRules
from src.reasoning_engine.forward_chainer import ForwardChainer
from src.reasoning_engine.explainer import ExplanationGenerator
from src.feature_engineering.calibrator import ThresholdCalibrator

In [2]:
# Load rules
rules = EconomicTheoryRules.get_instability_rules()
print(f"Loaded {len(rules)} rules:\n")
for rule in rules:
    print(f"{rule.id}: {rule.description}")
    print(f"  Antecedents: {rule.antecedents}")
    print(f"  Consequent: {rule.consequent}")
    print(f"  Confidence: {rule.confidence}\n")

Loaded 8 rules:

R1: Rising prices with high volume indicates speculation
  Antecedents: [{'HighVolume', 'RisingPrices'}]
  Consequent: SpeculativeActivity
  Confidence: 0.8

R2: Excessive gains with volume suggests irrational exuberance
  Antecedents: [{'HighVolume', 'ExcessiveGains'}]
  Consequent: SpeculativeActivity
  Confidence: 0.75

R3: Speculation with high volatility indicates bubble formation
  Antecedents: [{'SpeculativeActivity', 'HighVolatility'}]
  Consequent: BubbleState
  Confidence: 0.85

R4: Bubble with tight liquidity creates high crash risk
  Antecedents: [{'TightLiquidity', 'BubbleState'}]
  Consequent: HighCrashRisk
  Confidence: 0.9

R5: Falling prices with high volatility and tight liquidity indicates panic
  Antecedents: [{'HighVolatility', 'FallingPrices', 'TightLiquidity'}]
  Consequent: PanicState
  Confidence: 0.85

R6: Panic with low volume indicates liquidity crisis
  Antecedents: [{'LowVolume', 'PanicState'}]
  Consequent: LiquidityCrisis
  Confidence: 0

In [3]:
# Test individual rules
chainer = ForwardChainer(rules)

# Test case 1: RisingPrices + HighVolume (should trigger R1)
print("Test Case 1: RisingPrices + HighVolume")
facts1 = {'RisingPrices', 'HighVolume'}
result1 = chainer.infer(facts1)
print(f"Fired rules: {result1['fired_rules']}")
print(f"Derived facts: {result1['derived_facts']}")
print(f"Inference chain: {result1['inference_chain']}\n")

Test Case 1: RisingPrices + HighVolume
Fired rules: ['R1']
Derived facts: {'SpeculativeActivity'}
Inference chain: [('R1', 'SpeculativeActivity')]



In [4]:
# Test case 2: Full chain to HighCrashRisk
print("Test Case 2: Full chain to HighCrashRisk")
facts2 = {'RisingPrices', 'HighVolume', 'HighVolatility', 'TightLiquidity'}
result2 = chainer.infer(facts2)
print(f"Fired rules: {result2['fired_rules']}")
print(f"Derived facts: {result2['derived_facts']}")
print(f"Instability level: {result2['instability_level']}")
print(f"Inference chain:")
for rule_id, consequent in result2['inference_chain']:
    print(f"  [{rule_id}] → {consequent}")

Test Case 2: Full chain to HighCrashRisk
Fired rules: ['R1', 'R3', 'R4']
Derived facts: {'SpeculativeActivity', 'HighCrashRisk', 'BubbleState'}
Instability level: Medium
Inference chain:
  [R1] → SpeculativeActivity
  [R3] → BubbleState
  [R4] → HighCrashRisk
